In [23]:
import pandas as pd
import numpy as np
import ast

# 1. upload and last filter for last 10 years, from 2016
df = pd.read_csv('../data/raw/track_data_final.csv')
df['release_year'] = pd.to_datetime(df['album_release_date'], errors='coerce').dt.year
df_last_10_years = df[df['release_year'] >= 2016].copy()

# 2. Cleaning / Category-assignment function for a single song
def classify_main_genre(genre_val):
    
    genre_str = str(genre_val).lower()
    
    # if empyt, or '[]'
    if pd.isna(genre_val) or genre_val == 'N/A' or genre_val == '[]':
        return ""
        # return "unknown / no genre"
    # Keyword search within the line (if any match, this will be the category)
    if "hip hop" in genre_str or "rap" in genre_str or "trap" in genre_str or "drill" in genre_str:
        return "hip-Hop / rap"
    elif "pop" in genre_str:
        return "pop"
    elif "rock" in genre_str or "metal" in genre_str or "grunge" in genre_str or "punk" in genre_str:
        return "rock / metal"
    elif "latin" in genre_str or "reggaeton" in genre_str or "corridos" in genre_str or "sierraño" in genre_str:
        return "latin"
    elif "edm" in genre_str or "house" in genre_str or "techno" in genre_str or "electro" in genre_str or "trance" in genre_str or "phonk" in genre_str:
        return "electronic / edm"
    elif "r&b" in genre_str or "soul" in genre_str:
        return "r&b / soul"
    elif "indie" in genre_str or "alternative" in genre_str or "folk" in genre_str:
        return "indie / alternative"
    elif "classical" in genre_str or "ambient" in genre_str or "instrumental" in genre_str:
        return "classical / instrumental"
    elif "country" in genre_str or "acoustic country" in genre_str :
        return "country"
    elif "soundtrack" in genre_str or "anime" in genre_str :
        return "soundtrack"    
    else:
        return "other / niche"

# 3. Apply classify_classify_main_genre
df_last_10_years['main_genre'] = df_last_10_years['artist_genres'].apply(classify_main_genre)

# Verification: check whether the number of rows matches that of the filtered data (no explode)
print(f"Total tracks in last 10 years (rows preserved): {len(df_last_10_years)}")
print("-" * 60)
print("-" * 60)
# 4. Quick statistics for the new, clean main categories, per song
genre_counts = df_last_10_years['main_genre'].value_counts()
print("--- Track Counts per Main Genre ---")
print(genre_counts)

print("-" * 60)
# Filter out the rows that fall into the 'Other / Niche' category.
other_genres = df_last_10_years[df_last_10_years['main_genre'] == 'other / niche']
print("-" * 60)
print("List of uncategorized (Other / Niche) original genres (Top 30):")
display(other_genres['artist_genres'].value_counts().head(30))
# other_genres = df_exploded[df_exploded['high_level_genre'] == 'Other / Niche']


Total tracks in last 10 years (rows preserved): 5256
------------------------------------------------------------
------------------------------------------------------------
--- Track Counts per Main Genre ---
main_genre
unkonwn / no genre          3026
Pop                          702
hip-Hop / rap                543
electronic / edm             240
soundtrack                   179
r&b / soul                   164
other / niche                164
country                       92
rock / metal                  52
latin                         50
indie / alternative           25
classical / instrumental      19
Name: count, dtype: int64
------------------------------------------------------------
------------------------------------------------------------
List of uncategorized (Other / Niche) original genres (Top 30):


artist_genres
['medieval']                                   20
['nightcore']                                  17
['southern gothic']                            15
['moombahton']                                  7
['future bass']                                 5
['afrobeats']                                   5
['jersey club']                                 4
['riot grrrl', 'queercore']                     4
['melodic bass', 'nightcore', 'bass music']     4
['drum and bass']                               3
['dancehall']                                   3
['neo-psychedelic']                             3
['medieval', 'celtic']                          3
['singer-songwriter']                           3
['italo dance']                                 2
['comedy']                                      2
['synthwave', 'chillwave', 'vaporwave']         2
['big room']                                    2
['melodic bass', 'future bass']                 2
['alté']                            

In [26]:
df_last_10_years['main_genre'] = df_last_10_years['artist_genres'].apply(classify_main_genre)

print(f"Total track rows preserved after classification: {len(df_last_10_years)}")
print("-" * 60)

# 3. Standard statistics and averages for Track Popularity per Main Genre
print("--- Track Popularity Statistics by Main Genre (Last 10 Years) ---")
genre_popularity_stats = df_last_10_years.groupby('main_genre')['track_popularity'].agg(
    track_count='count',
    mean_popularity='mean',
    median_popularity='median',
    min_popularity='min',
    max_popularity='max',
    std_popularity='std'
).reset_index()

# Sort by mean popularity descending
genre_popularity_stats = genre_popularity_stats.sort_values(by='mean_popularity', ascending=False)

# Display top genres by average track popularity
display(genre_popularity_stats.head(15))

print("-" * 60)
print("--- Summary Describe for Track Popularity overall (Last 10 Years) ---")
display(df_last_10_years['track_popularity'].describe())

Total track rows preserved after classification: 5256
------------------------------------------------------------
--- Track Popularity Statistics by Main Genre (Last 10 Years) ---


,main_genre,track_count,mean_popularity,median_popularity,min_popularity,max_popularity,std_popularity
6,latin,50,64.560000,70.0,1,87,20.127309
5,indie / alternative,25,63.640000,69.0,27,85,17.672672
4,hip-Hop / rap,543,58.753223,65.0,0,91,21.915846
0,Pop,702,57.059829,62.0,0,99,22.739845
8,r&b / soul,164,52.750000,55.0,0,89,21.815561
11,unkonwn / no genre,3026,52.161269,59.0,0,100,24.242394
2,country,92,51.771739,48.0,0,86,16.356001
3,electronic / edm,240,48.233333,50.5,0,92,23.151577
9,rock / metal,52,41.730769,45.5,0,83,23.615478
7,other / niche,164,34.457317,35.0,0,85,23.539902


------------------------------------------------------------
--- Summary Describe for Track Popularity overall (Last 10 Years) ---


count    5256.000000
mean       52.124429
std        23.954884
min         0.000000
25%        38.000000
50%        58.000000
75%        70.000000
max       100.000000
Name: track_popularity, dtype: float64

In [27]:

# 5. Most frequent genres (sorted by track_count descending)
sorted_by_frequency = genre_popularity_stats.sort_values(by='track_count', ascending=False)

print("--- Top Genres Sorted by Frequency (Track Count - Last 10 Years) ---")
display(sorted_by_frequency.head(15))

print("-" * 60)

# 6. Most popular genres with a minimum occurrence threshold (at least x tracks)
min_count_threshold = 100 

popular_frequent_genres = genre_popularity_stats[genre_popularity_stats['track_count'] >= min_count_threshold]
popular_frequent_genres = popular_frequent_genres.sort_values(by='mean_popularity', ascending=False)

print(f"--- Top Popular Genres (appearing at least {min_count_threshold} times - Last 10 Years) ---")
display(popular_frequent_genres.head(15))

--- Top Genres Sorted by Frequency (Track Count - Last 10 Years) ---


,main_genre,track_count,mean_popularity,median_popularity,min_popularity,max_popularity,std_popularity
11,unkonwn / no genre,3026,52.161269,59.0,0,100,24.242394
0,Pop,702,57.059829,62.0,0,99,22.739845
4,hip-Hop / rap,543,58.753223,65.0,0,91,21.915846
3,electronic / edm,240,48.233333,50.5,0,92,23.151577
10,soundtrack,179,33.469274,37.0,0,71,16.968341
8,r&b / soul,164,52.750000,55.0,0,89,21.815561
7,other / niche,164,34.457317,35.0,0,85,23.539902
2,country,92,51.771739,48.0,0,86,16.356001
9,rock / metal,52,41.730769,45.5,0,83,23.615478
6,latin,50,64.560000,70.0,1,87,20.127309


------------------------------------------------------------
--- Top Popular Genres (appearing at least 100 times - Last 10 Years) ---


,main_genre,track_count,mean_popularity,median_popularity,min_popularity,max_popularity,std_popularity
4,hip-Hop / rap,543,58.753223,65.0,0,91,21.915846
0,Pop,702,57.059829,62.0,0,99,22.739845
8,r&b / soul,164,52.750000,55.0,0,89,21.815561
11,unkonwn / no genre,3026,52.161269,59.0,0,100,24.242394
3,electronic / edm,240,48.233333,50.5,0,92,23.151577
7,other / niche,164,34.457317,35.0,0,85,23.539902
10,soundtrack,179,33.469274,37.0,0,71,16.968341


In [17]:
display(df_last_10_years)

,track_id,track_name,track_number,track_popularity,track_duration_ms,explicit,artist_name,artist_popularity,artist_followers,artist_genres,album_id,album_name,album_release_date,album_total_tracks,album_type,release_year,main_genre
1,2lWc1iJlz2NVcStV5fbtPG,Clouds,1,67,158760,False,BUNT.,69.0,293734.0,['stutter house'],2ArRQNLxf9t0O0gvmG5Vsj,Clouds,2023-01-13,1,single,2023.0,electronic / edm
2,1msEuwSBneBKpVCZQcFTsU,Forever & Always (Taylor’s Version),11,63,225328,False,Taylor Swift,100.0,145396321.0,[],4hDok0OAJd57SGIT8xuWJH,Fearless (Taylor's Version),2021-04-09,26,album,2021.0,other / niche
3,7bcy34fBT2ap1L4bfPsl9q,I Didn't Change My Number,2,72,158463,True,Billie Eilish,90.0,118692183.0,[],0JGOiO34nwfUdDrD612dOp,Happier Than Ever,2021-07-30,16,album,2021.0,other / niche
6,41zXlQxzTi6cGAjpOXyLYH,idontwannabeyouanymore,2,78,203569,False,Billie Eilish,90.0,118692183.0,[],7fRrTyKvE4Skh93v97gtcU,dont smile at me,2017-12-22,9,single,2017.0,other / niche
7,13jRFAGT8qd6aBwtJySlUm,Allein Allein - BENNETT Remix,1,52,145977,False,Alok,76.0,11247155.0,"['brazilian bass', 'electronic', 'slap house',...",1WKoDELzbFRR6UWNGh50LO,Allein Allein (feat. FREY) [BENNETT Remix],2024-09-27,2,single,2024.0,electronic / edm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8769,4y4spB9m0Q6026KfkAvy9Q,Lonely (with benny blanco),1,66,149297,True,Justin Bieber,91.0,85050158.0,[],3P5WIUJO0Ots1lQx09TOxk,Lonely (with benny blanco),2020-10-16,1,single,2020.0,other / niche
8770,6GFrELGVKBXRcrTNkwjSfF,Body And Mind,3,54,186672,False,girl in red,73.0,8771418.0,['bedroom pop'],69p42ovQdwmYPLUsAQpeNn,if i could make it go quiet,2021-04-30,11,album,2021.0,Pop
8773,58VTP6KnZs12PcAj5rMJ4W,the fruits,1,65,209698,False,Paris Paloma,66.0,786890.0,[],48ssTnnFOCIP17sRaFNqu1,the fruits,2022-04-29,1,single,2022.0,other / niche
8774,2dRfUOeTVpr4V4PmmqGkpm,Dark World Theme (The Legend of Zelda),4,2,113143,False,Zelda,8.0,3263.0,[],0JJAfVZtn3EvZnXSQrpLSh,The Legend of Zelda,2023-08-19,7,album,2023.0,other / niche
